# Lookzi — AI Virtual Try-On (Google Colab)

**Birinchi ishga tushirishdan oldin:** `Runtime → Change runtime type → T4 GPU`

| Qadam | Nima qiladi | 1-marta | Keyingi marta |
|-------|-------------|---------|---------------|
| 1 | Google Drive ulash | ~10 sek | ~10 sek |
| 2 | GPU tekshirish | — | — |
| 3 | Repo + paketlar | ~5-10 daq | ~1-2 daq (cache) |
| 4 | Model weights | ~10-15 daq | **0 sek (Drive'da)** |
| 5 | Ishga tushirish | — | — |

> Hamma narsa `Google Drive/Lookzi/` ga saqlanadi — keyingi sessiyalarda qayta yuklanmaydi.

**Render vaqti (T4 GPU):** ~50-70 sekund (Steps=20), ~80-100 sekund (Steps=30)

In [ ]:
# ── Qadam 1: Google Drive ulash ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

LOOKZI_DRIVE = '/content/drive/MyDrive/Lookzi'
PIP_CACHE    = f'{LOOKZI_DRIVE}/pip_cache'
WEIGHTS_DIR  = f'{LOOKZI_DRIVE}/weights'
HF_CACHE     = f'{LOOKZI_DRIVE}/hf_cache'

for d in [LOOKZI_DRIVE, PIP_CACHE, WEIGHTS_DIR, HF_CACHE]:
    os.makedirs(d, exist_ok=True)

print('Drive ulandi!')
print(f'Papka: {LOOKZI_DRIVE}')
!ls -lh "{LOOKZI_DRIVE}"

In [ ]:
# ── Qadam 2: GPU tekshirish ───────────────────────────────────────────────
!nvidia-smi

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU     : {p.name}')
    print(f'VRAM    : {p.total_memory / 1e9:.1f} GB')

    # Qaysi dtype ishlatilishini ko'rsatish
    if torch.cuda.is_bf16_supported():
        dtype = 'bfloat16  (eng tez)'
    else:
        dtype = 'float16   (T4 uchun optimallashtirilgan)'
    print(f'Dtype   : {dtype}')
else:
    print('\nGPU topilmadi! Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Qadam 3: Repo clone + paketlar (pip cache Drive'da) ──────────────────
import os, sys

LOOKZI_DRIVE = '/content/drive/MyDrive/Lookzi'
PIP_CACHE    = f'{LOOKZI_DRIVE}/pip_cache'
REPO_DIR     = '/content/Lookzi-v0.1'

# Repo
if os.path.exists(REPO_DIR):
    print('Repo mavjud — yangilanmoqda...')
    !git -C "{REPO_DIR}" pull --quiet
else:
    print('Repo yuklanmoqda...')
    !git clone https://github.com/Mohamed-Kudratov/Lookzi-v0.1.git "{REPO_DIR}" --quiet

os.chdir(REPO_DIR)

# Paketlar (cache Drive'da — keyingi safar tez)
print('\nPaketlar o\'rnatilmoqda...')
!pip install . --cache-dir="{PIP_CACHE}" -q
!pip install 'fastapi>=0.100' 'uvicorn[standard]' 'gradio>=6.0' --cache-dir="{PIP_CACHE}" -q

import importlib; importlib.invalidate_caches()

import torch, gradio, fashn_vton
print(f'Torch  : {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'Gradio : {gradio.__version__}')
print('fashn_vton: OK')
print('\nHamma paketlar tayyor!')

In [ ]:
# ── Qadam 4: Model weights (Drive'da saqlanadi, bir marta) ────────────────
import os

LOOKZI_DRIVE = '/content/drive/MyDrive/Lookzi'
WEIGHTS_DIR  = f'{LOOKZI_DRIVE}/weights'
HF_CACHE     = f'{LOOKZI_DRIVE}/hf_cache'
REPO_DIR     = '/content/Lookzi-v0.1'

os.environ['HF_HOME'] = HF_CACHE
os.chdir(REPO_DIR)

# weights/ papkasini Drive'ga symlink
weights_link = f'{REPO_DIR}/weights'
if os.path.islink(weights_link):
    os.unlink(weights_link)
if not os.path.exists(weights_link):
    os.symlink(WEIGHTS_DIR, weights_link)

model_file = f'{WEIGHTS_DIR}/model.safetensors'
yolox_file = f'{WEIGHTS_DIR}/dwpose/yolox_l.onnx'

if os.path.exists(model_file) and os.path.exists(yolox_file):
    gb = os.path.getsize(model_file) / 1e9
    print(f'Weights Drive\'da mavjud ({gb:.2f} GB) — qayta yuklanmaydi!')
    !ls -lh "{WEIGHTS_DIR}/"
    !ls -lh "{WEIGHTS_DIR}/dwpose/"
else:
    print('Weights yuklanmoqda (~3 GB, faqat 1-marta)...')
    !python scripts/download_weights.py --weights-dir "{WEIGHTS_DIR}"
    print('\nWeights Drive\'ga saqlandi — keyingi safar bu qadam 0 sekund!')

In [ ]:
# ── Qadam 5: Lookzi ishga tushirish ───────────────────────────────────────
# Public share URL quyida paydo bo'ladi (72 soat ishlaydi)
import sys, os

LOOKZI_DRIVE = '/content/drive/MyDrive/Lookzi'
REPO_DIR     = '/content/Lookzi-v0.1'

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.environ.update({
    'HF_HUB_DISABLE_SYMLINKS_WARNING': '1',
    'TOKENIZERS_PARALLELISM'          : 'false',
    'HF_HOME'                         : f'{LOOKZI_DRIVE}/hf_cache',
})

import app as lookzi_app

# T4 uchun default steps ni 20 ga tushirish (35s local = 50-70s Colab)
import torch
if torch.cuda.is_available() and not torch.cuda.is_bf16_supported():
    lookzi_app.DEFAULT_STEPS = 20
    print('T4 GPU aniqlandi — Steps 20 ga o\'rnatildi (tezroq render)')

# Model yuklash
print('Model yuklanmoqda...')
pipe = lookzi_app.get_pipeline()
print(f'Model tayyor: {pipe.device} | dtype: {pipe.inference_dtype}')

# UI ishga tushirish
demo = lookzi_app.build_ui()
demo.launch(
    share=True,
    debug=False,
    show_error=True,
)